In [ ]:
# Course Concept Map — ESS 412/512 Synthesis Figure
# ---------------------------------------------------
# Generates a visual map showing how the 8 course modules
# connect, with arrows pointing toward frontier topics.
#
# Output: figures/fig_course_concept_map.png (.pdf)

from __future__ import annotations

import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

plt.rcParams.update({
    "figure.dpi": 160,
    "savefig.dpi": 200,
    "font.size": 10,
})

OUTDIR = "figures"
os.makedirs(OUTDIR, exist_ok=True)


def _save(fig, fname):
    for ext in ("png", "pdf"):
        path = os.path.join(OUTDIR, f"{fname}.{ext}")
        fig.savefig(path, bbox_inches="tight", format=ext)
        print(f"Saved: {path}")
    plt.close(fig)

In [ ]:
# =======================================================================
# Figure: Course concept map
# =======================================================================

fig, ax = plt.subplots(figsize=(12, 7))
ax.set_xlim(-1, 13)
ax.set_ylim(-1, 8.5)
ax.axis("off")
ax.set_title("ESS 412/512 — Course Concept Map", fontsize=14,
             fontweight="bold", pad=12)

# --- Module boxes: (x, y, label, color) ---
modules = {
    "M1": (0.5, 6.5, "M1: Data &\nFourier", "#AED6F1"),
    "M2": (0.5, 4.0, "M2: Stress &\nStrain", "#A9DFBF"),
    "M3": (3.5, 5.2, "M3: Body\nWaves", "#F9E79F"),
    "M4": (3.5, 7.2, "M4: Reflection\nSeismology", "#F5CBA7"),
    "M5": (3.5, 3.0, "M5: Surface\nWaves", "#D7BDE2"),
    "M6": (6.5, 6.5, "M6: Earthquake\nLocation", "#F1948A"),
    "M7": (6.5, 4.0, "M7: Moment\nTensors &\nMagnitudes", "#F5B7B1"),
    "M8": (9.5, 5.2, "M8: Nucleation\n& Dynamics", "#E8DAEF"),
}

box_w, box_h = 2.2, 1.3
box_patches = {}

for key, (x, y, label, color) in modules.items():
    bx = x - box_w / 2
    by = y - box_h / 2
    p = FancyBboxPatch((bx, by), box_w, box_h,
                       boxstyle="round,pad=0.1",
                       fc=color, ec="0.3", lw=1.2, zorder=3)
    ax.add_patch(p)
    ax.text(x, y, label, ha="center", va="center", fontsize=9,
            fontweight="bold", zorder=4)
    box_patches[key] = (x, y)

# --- Arrows connecting modules ---
# (from, to, label)
connections = [
    ("M1", "M3", "signals"),
    ("M1", "M4", ""),
    ("M2", "M3", "elasticity"),
    ("M2", "M5", "continuum\nmechanics"),
    ("M3", "M4", "ray\ntheory"),
    ("M3", "M5", "dispersion"),
    ("M3", "M6", "travel\ntimes"),
    ("M5", "M7", "spectra"),
    ("M6", "M7", "source"),
    ("M7", "M8", "energy &\nstress drop"),
    ("M6", "M8", "fault\ngeometry"),
    ("M2", "M8", "friction"),
]

for src, dst, lbl in connections:
    sx, sy = box_patches[src]
    dx, dy = box_patches[dst]
    # Shorten arrows so they don't overlap boxes
    angle = np.arctan2(dy - sy, dx - sx)
    offset_s = 0.7
    offset_d = 0.7
    ax.annotate("",
                xy=(dx - offset_d * np.cos(angle),
                    dy - offset_d * np.sin(angle)),
                xytext=(sx + offset_s * np.cos(angle),
                        sy + offset_s * np.sin(angle)),
                arrowprops=dict(arrowstyle="->", lw=1.2, color="0.4",
                               connectionstyle="arc3,rad=0.1"),
                zorder=1)
    if lbl:
        mx = (sx + dx) / 2
        my = (sy + dy) / 2
        ax.text(mx, my + 0.15, lbl, fontsize=7, ha="center", va="center",
                color="0.35", fontstyle="italic", zorder=2)

# --- Frontier arrows (from M8 region, pointing right/down) ---
frontiers = [
    (11.2, 7.5, "ML / AI"),
    (11.2, 6.3, "DAS"),
    (11.2, 5.1, "EEW"),
    (11.2, 3.9, "Slow EQs"),
    (11.2, 2.7, "Planetary\nSeismology"),
]

# Bracket region for frontiers
frontier_box = FancyBboxPatch((10.8, 2.2), 2.0, 6.0,
                              boxstyle="round,pad=0.15",
                              fc="#FDEBD0", ec="C1", lw=1.5,
                              ls="--", zorder=0, alpha=0.5)
ax.add_patch(frontier_box)
ax.text(11.8, 8.0, "Frontiers", fontsize=11, ha="center",
        fontweight="bold", color="C1")

for fx, fy, flbl in frontiers:
    ax.text(fx + 0.6, fy, flbl, fontsize=9, ha="center", va="center",
            color="0.3", fontweight="bold", zorder=4)

# Arrow from M8 to frontier region
ax.annotate("", xy=(10.6, 5.2), xytext=(10.1, 5.2),
            arrowprops=dict(arrowstyle="->", lw=2, color="C1"),
            zorder=5)

# --- Recurring themes bar at bottom ---
themes = ["Fourier analysis", "Inverse problems", "Eigenvalue problems",
          "Energy methods", "Continuum mechanics"]
theme_x = np.linspace(1.0, 11.0, len(themes))
for tx, tl in zip(theme_x, themes):
    ax.text(tx, 0.8, tl, fontsize=8, ha="center", va="center",
            bbox=dict(boxstyle="round,pad=0.25", fc="#EBF5FB", ec="0.6",
                      lw=0.8),
            color="0.3", fontstyle="italic")

ax.text(6.0, 0.0, "Recurring mathematical themes", fontsize=10,
        ha="center", va="center", fontweight="bold", color="0.4")

# Dashed line separating themes from modules
ax.plot([-0.5, 12.5], [1.5, 1.5], "--", color="0.7", lw=0.8)

fig.tight_layout()
_save(fig, "fig_course_concept_map")

In [ ]:
print("Synthesis figure generated successfully.")